In [1]:
import pandas as pd

df = pd.read_parquet("../data/history/all_seasons_fixed.parquet")

print("=== SHAPE ===")
print(df.shape)

print("\n=== COLUMNS ===")
print(list(df.columns))

print("\n=== SEASONS ===")
print(df["season"].value_counts().sort_index())

=== SHAPE ===
(253900, 74)

=== COLUMNS ===
['name', 'assists', 'attempted_passes', 'big_chances_created', 'big_chances_missed', 'bonus', 'bps', 'clean_sheets', 'clearances_blocks_interceptions', 'completed_passes', 'creativity', 'dribbles', 'ea_index', 'element', 'errors_leading_to_goal', 'errors_leading_to_goal_attempt', 'fixture', 'fouls', 'goals_conceded', 'goals_scored', 'ict_index', 'id', 'influence', 'key_passes', 'kickoff_time', 'kickoff_time_formatted', 'loaned_in', 'loaned_out', 'minutes', 'offside', 'open_play_crosses', 'opponent_team', 'own_goals', 'penalties_conceded', 'penalties_missed', 'penalties_saved', 'recoveries', 'red_cards', 'round', 'saves', 'selected', 'tackled', 'tackles', 'target_missed', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'winning_goals', 'yellow_cards', 'GW', 'position', 'team', 'season', 'xP', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 

In [2]:
# Sort first — this is non-negotiable for shift/rolling to be correct
df = df.sort_values(["season", "element", "round"]).reset_index(drop=True)

# Build ONE rolling feature, leakage-safe:
# groupby(season, element) so it never crosses players or seasons
# shift(1) removes the current game BEFORE averaging
# rolling(3) averages the 3 games before this one
df["pts_last3"] = (
    df.groupby(["season", "element"])["total_points"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# Eyeball it: pick one high-profile player in one season
check = df[(df["season"] == "2025-26") & (df["name"].str.contains("Haaland", case=False, na=False))]
print(check[["name", "round", "total_points", "pts_last3"]].head(8).to_string(index=False))

          name  round  total_points  pts_last3
Erling Haaland      1            13        NaN
Erling Haaland      2             2  13.000000
Erling Haaland      3             9   7.500000
Erling Haaland      4            13   8.000000
Erling Haaland      5             9   8.000000
Erling Haaland      6            16  10.333333
Erling Haaland      7             8  12.666667
Erling Haaland      8            13  11.000000


In [3]:
# Columns to roll (the raw ingredients our components used)
roll_cols = [
    "total_points",
    "minutes",
    "expected_goals",
    "expected_assists",
    "bps",
    "starts",
]

windows = [3, 5]

# Same proven recipe: sort, groupby(player+season), shift(1), rolling.mean()
# (already sorted above, but re-sort to be safe/explicit)
df = df.sort_values(["season", "element", "round"]).reset_index(drop=True)

for col in roll_cols:
    for w in windows:
        df[f"{col}_last{w}"] = (
            df.groupby(["season", "element"])[col]
            .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        )

# Show the new feature columns for Haaland to sanity-check
new_cols = [f"{c}_last{w}" for c in roll_cols for w in windows]
check = df[(df["season"] == "2025-26") & (df["name"].str.contains("Haaland", case=False, na=False))]
print(check[["name", "round", "total_points"] + new_cols].head(6).to_string(index=False))

          name  round  total_points  total_points_last3  total_points_last5  minutes_last3  minutes_last5  expected_goals_last3  expected_goals_last5  expected_assists_last3  expected_assists_last5  bps_last3  bps_last5  starts_last3  starts_last5
Erling Haaland      1            13                 NaN                 NaN            NaN            NaN                   NaN                   NaN                     NaN                     NaN        NaN        NaN           NaN           NaN
Erling Haaland      2             2           13.000000               13.00      72.000000           72.0              1.990000              1.990000                0.000000                  0.0000  49.000000      49.00           1.0           1.0
Erling Haaland      3             9            7.500000                7.50      81.000000           81.0              1.230000              1.230000                0.150000                  0.1500  27.500000      27.50           1.0           1.0
Erling H

In [4]:
# was_home is already 0/1 (or True/False) — cast to int to be safe
df["was_home"] = df["was_home"].astype(int)

# Static attributes (known before the match, so legal as-is):
# position and value(price) are point-in-time correct in this file
# position is a string (GK/DEF/MID/FWD) — we'll encode it for LightGBM shortly
print(df[["name", "round", "was_home", "position", "value"]].head(5).to_string(index=False))
print("\nPositions present:", df["position"].unique())
print("Value (price) range:", df["value"].min(), "to", df["value"].max())

        name  round  was_home position  value
David_Ospina      1         1       GK     50
David_Ospina      2         0       GK     50
David_Ospina      3         0       GK     50
David_Ospina      4         1       GK     49
David_Ospina      5         0       GK     49

Positions present: <ArrowStringArray>
['GK', 'DEF', 'MID', 'FWD', 'GKP', 'AM']
Length: 6, dtype: str
Value (price) range: 5 to 154


In [5]:
# How common are the odd position labels, and in which seasons?
print("=== Position counts ===")
print(df["position"].value_counts())

print("\n=== Which seasons use GKP vs GK? ===")
print(df.groupby("season")["position"].apply(lambda s: sorted(s.unique())).to_string())

print("\n=== What is 'AM'? Show a few rows ===")
print(df[df["position"] == "AM"][["name", "season", "round", "position"]].head(10).to_string(index=False))

print("\n=== Suspicious low prices (value < 35 = under £3.5m) ===")
print("Count:", (df["value"] < 35).sum())
print(df[df["value"] < 35][["name", "season", "round", "value", "minutes"]].head(10).to_string(index=False))

=== Position counts ===
position
MID    107747
DEF     85284
FWD     32780
GK      27666
AM        322
GKP       101
Name: count, dtype: int64

=== Which seasons use GKP vs GK? ===
season
2016-17         [DEF, FWD, GK, MID]
2017-18         [DEF, FWD, GK, MID]
2018-19         [DEF, FWD, GK, MID]
2019-20         [DEF, FWD, GK, MID]
2020-21         [DEF, FWD, GK, MID]
2021-22    [DEF, FWD, GK, GKP, MID]
2022-23         [DEF, FWD, GK, MID]
2023-24         [DEF, FWD, GK, MID]
2024-25     [AM, DEF, FWD, GK, MID]
2025-26         [DEF, FWD, GK, MID]

=== What is 'AM'? Show a few rows ===
        name  season  round position
Mikel Arteta 2024-25     23       AM
Mikel Arteta 2024-25     24       AM
Mikel Arteta 2024-25     25       AM
Mikel Arteta 2024-25     26       AM
Mikel Arteta 2024-25     27       AM
Mikel Arteta 2024-25     28       AM
Mikel Arteta 2024-25     29       AM
Mikel Arteta 2024-25     30       AM
Mikel Arteta 2024-25     31       AM
Mikel Arteta 2024-25     32       AM

=== S

In [6]:
# 1. Drop Assistant Managers — not players, different scoring rules
before = len(df)
df = df[df["position"] != "AM"].copy()
print(f"Dropped {before - len(df)} AM (manager) rows")

# 2. Merge GKP into GK — same position, label drift in 2021-22
df["position"] = df["position"].replace({"GKP": "GK"})

# Confirm we now have exactly 4 clean positions
print("Positions now:", sorted(df["position"].unique()))
print("Counts:\n", df["position"].value_counts().to_string())

Dropped 322 AM (manager) rows
Positions now: ['DEF', 'FWD', 'GK', 'MID']
Counts:
 position
MID    107747
DEF     85284
FWD     32780
GK      27767


In [7]:
# Look at the suspicious low-price rows in detail
low = df[df["value"] < 35]
print("Total low-price rows (< £3.5m):", len(low))
print("\nBy season:")
print(low.groupby("season").size().to_string())
print("\nMinutes distribution of these rows:")
print("  0 minutes:", (low["minutes"] == 0).sum())
print("  played (>0 min):", (low["minutes"] > 0).sum())
print("\nSample rows:")
print(low[["name", "season", "round", "value", "minutes", "total_points"]].head(10).to_string(index=False))

Total low-price rows (< £3.5m): 0

By season:
Series([], )

Minutes distribution of these rows:
  0 minutes: 0
  played (>0 min): 0

Sample rows:
Empty DataFrame
Columns: [name, season, round, value, minutes, total_points]
Index: []


In [8]:
print("Value (price) range now:", df["value"].min(), "to", df["value"].max())
print("\nLowest 5 prices:")
print(df.nsmallest(5, "value")[["name", "season", "value", "minutes"]].to_string(index=False))

Value (price) range now: 36 to 154

Lowest 5 prices:
                 name  season  value  minutes
    Caoimhin Kelleher 2023-24     36        0
 Stefan Ortega Moreno 2023-24     36        0
 Stefan Ortega Moreno 2023-24     36       21
 Stefan Ortega Moreno 2023-24     36       90
Konstantinos Tsimikas 2021-22     37       90


In [9]:
# One-hot encode position: 4 columns, each 0/1, no fake ordering
pos_dummies = pd.get_dummies(df["position"], prefix="pos").astype(int)
df = pd.concat([df, pos_dummies], axis=1)

print("New position columns:", list(pos_dummies.columns))
print(df[["name", "position"] + list(pos_dummies.columns)].head(5).to_string(index=False))

New position columns: ['pos_DEF', 'pos_FWD', 'pos_GK', 'pos_MID']
        name position  pos_DEF  pos_FWD  pos_GK  pos_MID
David_Ospina       GK        0        0       1        0
David_Ospina       GK        0        0       1        0
David_Ospina       GK        0        0       1        0
David_Ospina       GK        0        0       1        0
David_Ospina       GK        0        0       1        0


In [10]:
# Feature list — the 17 legal, as-of features we built
feature_cols = [
    "total_points_last3", "total_points_last5",
    "minutes_last3", "minutes_last5",
    "expected_goals_last3", "expected_goals_last5",
    "expected_assists_last3", "expected_assists_last5",
    "bps_last3", "bps_last5",
    "starts_last3", "starts_last5",
    "was_home", "value",
    "pos_DEF", "pos_FWD", "pos_GK", "pos_MID",
]

target_col = "total_points"

# Split by season — 2025-26 sealed as test, matching the decomposed model
train = df[df["season"] != "2025-26"].copy()
test  = df[df["season"] == "2025-26"].copy()

print("Train rows:", len(train), "| seasons:", sorted(train['season'].unique()))
print("Test rows: ", len(test), "| season:", sorted(test['season'].unique()))
print("\nFeature count:", len(feature_cols))

Train rows: 223821 | seasons: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Test rows:  29757 | season: ['2025-26']

Feature count: 18


In [11]:
import lightgbm as lgb
print("LightGBM version:", lgb.__version__)

LightGBM version: 4.7.0


In [12]:
from scipy.stats import spearmanr

# Build the training matrices
X_train = train[feature_cols]
y_train = train[target_col]
X_test  = test[feature_cols]
y_test  = test[target_col]

# The model — deliberately untuned, sensible defaults for a fair benchmark
model = lgb.LGBMRegressor(
    n_estimators=300,      # 300 trees (correcting each other)
    learning_rate=0.05,    # each tree nudges gently (avoids overfitting)
    num_leaves=31,         # tree complexity — LightGBM's default
    random_state=42,       # reproducibility
    verbose=-1,            # silence the training chatter
)

model.fit(X_train, y_train)

# Predict the sealed test season
test_preds = model.predict(X_test)

# Score it — Spearman is the headline (matches the decomposed model's metric)
# Only score rows that have an actual result (drop any NaN targets, just in case)
mask = y_test.notna()
rank_corr, _ = spearmanr(test_preds[mask], y_test[mask])
mae = (abs(test_preds[mask] - y_test[mask])).mean()

print("=== LightGBM Direct Regression (Option B) — 2025-26 sealed test ===")
print(f"Spearman (rank): {rank_corr:.3f}")
print(f"MAE:             {mae:.3f}")
print(f"\nDecomposed model for comparison: Spearman 0.716, MAE 1.13")

=== LightGBM Direct Regression (Option B) — 2025-26 sealed test ===
Spearman (rank): 0.708
MAE:             0.995

Decomposed model for comparison: Spearman 0.716, MAE 1.13


In [13]:
# Re-score on minutes bands — where do the models actually differ?
test_scored = test.copy()
test_scored["pred"] = test_preds
test_scored = test_scored[test_scored[target_col].notna()]

bands = {
    "All rows":        test_scored[target_col].notna(),          # baseline (what we already have)
    "Played (>0 min)": test_scored["minutes"] > 0,
    "Started (60+)":   test_scored["minutes"] >= 60,
}

print(f"{'Slice':<18}{'N':>8}{'LGBM Spearman':>16}{'LGBM MAE':>12}")
print("-" * 54)
for label, m in bands.items():
    sub = test_scored[m]
    rc, _ = spearmanr(sub["pred"], sub[target_col])
    mae = (sub["pred"] - sub[target_col]).abs().mean()
    print(f"{label:<18}{len(sub):>8}{rc:>16.3f}{mae:>12.3f}")

Slice                    N   LGBM Spearman    LGBM MAE
------------------------------------------------------
All rows             29757           0.708       0.995
Played (>0 min)      11498           0.316       1.998
Started (60+)         7815           0.100       2.383


In [14]:
# Load the decomposed model's predictions
decomp = pd.read_parquet("../data/predictions_2526.parquet")

print("=== SHAPE ===")
print(decomp.shape)
print("\n=== COLUMNS ===")
print(list(decomp.columns))
print("\n=== HEAD ===")
print(decomp.head(3).to_string(index=False))

=== SHAPE ===
(29338, 16)

=== COLUMNS ===
['element', 'player_id', 'understat_id', 'gw', 'name', 'position', 'team', 'e_minutes', 'e_points', 'e_points_core', 'exp_bonus', 'pts_goals', 'pts_assists', 'pts_cs', 'pts_dc', 'pts_appear']

=== HEAD ===
 element  player_id  understat_id  gw              name position    team  e_minutes  e_points  e_points_core  exp_bonus  pts_goals  pts_assists   pts_cs  pts_dc  pts_appear
       1          1        9676.0   1 David Raya Martín       GK Arsenal  57.036650  2.825260       2.641448   0.183812   0.277130     0.144664 0.864091     0.0    1.355562
       1          1        9676.0   2 David Raya Martín       GK Arsenal  79.936048  4.611181       4.352299   0.258881   0.675658     0.352699 1.523976     0.0    1.799967
       1          1        9676.0   3 David Raya Martín       GK Arsenal  82.790917  3.487996       3.220125   0.267871   0.289221     0.150976 0.917830     0.0    1.862099


In [15]:
# Build LightGBM's per-row prediction table for the test season, with keys
lgbm_out = test[["element", "round", "minutes", target_col]].copy()
lgbm_out = lgbm_out.rename(columns={"round": "gw", target_col: "actual"})
lgbm_out["pred_lgbm"] = test_preds  # test_preds aligns row-for-row with test

# Decomposed predictions, keyed the same way
decomp_out = decomp[["element", "gw", "e_points"]].rename(columns={"e_points": "pred_decomp"})

# Inner join → only player-GWs BOTH models predicted AND that have an actual
merged = lgbm_out.merge(decomp_out, on=["element", "gw"], how="inner")
merged = merged[merged["actual"].notna()]

print("Shared player-GWs scored by BOTH models:", len(merged))

# Identical three bands, both models head-to-head
bands = {
    "All shared":      merged["actual"].notna(),
    "Played (>0 min)": merged["minutes"] > 0,
    "Started (60+)":   merged["minutes"] >= 60,
}

print(f"\n{'Slice':<18}{'N':>7}{'LGBM':>9}{'Decomp':>9}   (Spearman)")
print("-" * 50)
for label, m in bands.items():
    sub = merged[m]
    rc_l, _ = spearmanr(sub["pred_lgbm"],   sub["actual"])
    rc_d, _ = spearmanr(sub["pred_decomp"], sub["actual"])
    print(f"{label:<18}{len(sub):>7}{rc_l:>9.3f}{rc_d:>9.3f}")

Shared player-GWs scored by BOTH models: 29757

Slice                   N     LGBM   Decomp   (Spearman)
--------------------------------------------------
All shared          29757    0.708    0.714
Played (>0 min)     11498    0.316    0.341
Started (60+)        7815    0.100    0.104


In [16]:
print(f"{'Slice':<18}{'N':>7}{'LGBM MAE':>10}{'Decomp MAE':>12}")
print("-" * 47)
for label, m in bands.items():
    sub = merged[m]
    mae_l = (sub["pred_lgbm"]   - sub["actual"]).abs().mean()
    mae_d = (sub["pred_decomp"] - sub["actual"]).abs().mean()
    print(f"{label:<18}{len(sub):>7}{mae_l:>10.3f}{mae_d:>12.3f}")

Slice                   N  LGBM MAE  Decomp MAE
-----------------------------------------------
All shared          29757     0.995       1.126
Played (>0 min)     11498     1.998       1.998
Started (60+)        7815     2.383       2.361


In [17]:
print("Seasons in the decomposed predictions file:")
# does it even have a season column?
print("Columns:", list(decomp.columns))
if "season" in decomp.columns:
    print(decomp["season"].value_counts())
else:
    print("No 'season' column — checking gw range instead:")
    print("gw min/max:", decomp["gw"].min(), decomp["gw"].max())
    print("row count:", len(decomp))

Seasons in the decomposed predictions file:
Columns: ['element', 'player_id', 'understat_id', 'gw', 'name', 'position', 'team', 'e_minutes', 'e_points', 'e_points_core', 'exp_bonus', 'pts_goals', 'pts_assists', 'pts_cs', 'pts_dc', 'pts_appear']
No 'season' column — checking gw range instead:
gw min/max: 1 38
row count: 29338


In [18]:
# Actuals for 2025-26, keyed to match decomp
actuals = df[df["season"] == "2025-26"][["element", "round", "total_points", "minutes"]].copy()
actuals = actuals.rename(columns={"round": "gw", "total_points": "actual"})

# Join component predictions to actuals
a_df = decomp.merge(actuals, on=["element", "gw"], how="inner")
a_df = a_df[a_df["actual"].notna()]

print("Rows with components + actuals:", len(a_df))
print("GW range:", a_df["gw"].min(), "to", a_df["gw"].max())

# The component features for Option A
comp_features = ["e_minutes", "pts_goals", "pts_assists", "pts_cs", "pts_dc", "pts_appear", "exp_bonus"]
print("\nComponent features:", comp_features)
print("Any missing values?\n", a_df[comp_features].isna().sum())

Rows with components + actuals: 29757
GW range: 1 to 38

Component features: ['e_minutes', 'pts_goals', 'pts_assists', 'pts_cs', 'pts_dc', 'pts_appear', 'exp_bonus']
Any missing values?
 e_minutes      0
pts_goals      0
pts_assists    0
pts_cs         0
pts_dc         0
pts_appear     0
exp_bonus      0
dtype: int64


In [19]:
# Split within 2025-26: train GW1-25, test GW26-38
a_train = a_df[a_df["gw"] <= 25]
a_test  = a_df[a_df["gw"] >= 26]

print("Train rows (GW1-25):", len(a_train))
print("Test rows (GW26-38):", len(a_test))

# Train LightGBM on the COMPONENTS (not raw features)
model_a = lgb.LGBMRegressor(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    random_state=42, verbose=-1,
)
model_a.fit(a_train[comp_features], a_train["actual"])

a_test = a_test.copy()
a_test["pred_lgbm_A"] = model_a.predict(a_test[comp_features])

# Compare on the SAME test rows: LightGBM-A vs your equation (e_points)
print(f"\n{'Slice':<18}{'N':>7}{'LGBM-A':>9}{'Equation':>10}   (Spearman)")
print("-" * 48)
bands_a = {
    "All":            a_test["actual"].notna(),
    "Played (>0)":    a_test["minutes"] > 0,
    "Started (60+)":  a_test["minutes"] >= 60,
}
for label, m in bands_a.items():
    sub = a_test[m]
    rc_lgbm, _ = spearmanr(sub["pred_lgbm_A"], sub["actual"])
    rc_eq, _   = spearmanr(sub["e_points"],    sub["actual"])
    print(f"{label:<18}{len(sub):>7}{rc_lgbm:>9.3f}{rc_eq:>10.3f}")

Train rows (GW1-25): 19000
Test rows (GW26-38): 10757

Slice                   N   LGBM-A  Equation   (Spearman)
------------------------------------------------
All                 10757    0.753     0.713
Played (>0)          3918    0.287     0.353
Started (60+)        2670    0.109     0.125


In [20]:
print(f"{'Slice':<18}{'N':>7}{'LGBM-A MAE':>12}{'Equation MAE':>14}")
print("-" * 51)
for label, m in bands_a.items():
    sub = a_test[m]
    mae_lgbm = (sub["pred_lgbm_A"] - sub["actual"]).abs().mean()
    mae_eq   = (sub["e_points"]    - sub["actual"]).abs().mean()
    print(f"{label:<18}{len(sub):>7}{mae_lgbm:>12.3f}{mae_eq:>14.3f}")

Slice                   N  LGBM-A MAE  Equation MAE
---------------------------------------------------
All                 10757       0.873         1.072
Played (>0)          3918       2.071         1.968
Started (60+)        2670       2.386         2.327


In [21]:
try:
    import shap
    print("SHAP version:", shap.__version__)
except ImportError:
    print("Not installed — run: uv add shap")

c:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP version: 0.52.0


In [22]:
import shap

# TreeSHAP on the Option B model — fast and exact for trees
explainer = shap.TreeExplainer(model)

# Use a sample of the test set (SHAP on all 29k is slow and unnecessary for the picture)
X_sample = X_test.sample(2000, random_state=42)
shap_values = explainer.shap_values(X_sample)

# Global importance = mean absolute SHAP per feature
import numpy as np
importance = np.abs(shap_values).mean(axis=0)
imp_df = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap": importance
}).sort_values("mean_abs_shap", ascending=False)

print("=== Global feature importance (Option B / LightGBM) ===")
print(imp_df.to_string(index=False))

=== Global feature importance (Option B / LightGBM) ===
               feature  mean_abs_shap
         minutes_last3       0.664824
    total_points_last3       0.247612
                 value       0.232601
    total_points_last5       0.071253
              was_home       0.068107
         minutes_last5       0.060663
             bps_last3       0.045899
  expected_goals_last3       0.043502
                pos_GK       0.035764
               pos_DEF       0.028407
          starts_last3       0.027103
          starts_last5       0.026052
             bps_last5       0.021943
expected_assists_last3       0.021737
expected_assists_last5       0.020462
  expected_goals_last5       0.015942
               pos_MID       0.008633
               pos_FWD       0.006373
